In [4]:
import pandas as pd

from pathlib import Path

import joblib

BASE_DIR = Path.cwd().parent

DATA_PATH = BASE_DIR / "data" / "ai4i2020.csv"
MODEL_PATH = BASE_DIR / "models" / "random_forest_pipeline.joblib"

df = pd.read_csv(DATA_PATH)
model = joblib.load(MODEL_PATH)

FAILURE_THRESHOLD = 0.40

print("Dataset shape:", df.shape)
print("Model loaded:", type(model))

df[
    [
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ]
].agg(["min", "max"])

Dataset shape: (10000, 14)
Model loaded: <class 'sklearn.pipeline.Pipeline'>


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
min,295.3,305.7,1168,3.8,0
max,304.5,313.8,2886,76.6,253


In [2]:
test_columns = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Machine failure"
]

failure_examples = (
    df[df["Machine failure"] == 1][test_columns]
    .head(5)
)

non_failure_examples = (
    df[df["Machine failure"] == 0][test_columns]
    .head(5)
)

print("KNOWN FAILURE EXAMPLES")
display(failure_examples)

print("KNOWN NON-FAILURE EXAMPLES")
display(non_failure_examples)

KNOWN FAILURE EXAMPLES


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
50,L,298.9,309.1,2861,4.6,143,1
69,L,298.9,309.0,1410,65.7,191,1
77,L,298.8,308.9,1455,41.3,208,1
160,L,298.4,308.2,1282,60.7,216,1
161,L,298.3,308.1,1412,52.3,218,1


KNOWN NON-FAILURE EXAMPLES


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [5]:
test_examples = pd.concat([
    failure_examples,
    non_failure_examples
])

X_manual_test = test_examples.drop(columns="Machine failure")

test_results = test_examples.copy()

test_results["Failure Probability"] = (
    model.predict_proba(X_manual_test)[:, 1]
)

test_results["Predicted Failure"] = (
    test_results["Failure Probability"] >= 0.40
).astype(int)

test_results[
    [
        "Type",
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]",
        "Machine failure",
        "Failure Probability",
        "Predicted Failure"
    ]
]

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Failure Probability,Predicted Failure
50,L,298.9,309.1,2861,4.6,143,1,1.00,1
69,L,298.9,309.0,1410,65.7,191,1,1.00,1
77,L,298.8,308.9,1455,41.3,208,1,1.00,1
160,L,298.4,308.2,1282,60.7,216,1,1.00,1
161,L,298.3,308.1,1412,52.3,218,1,1.00,1
0,M,298.1,308.6,1551,42.8,0,0,0.00,0
1,L,298.2,308.7,1408,46.3,3,0,0.01,0
2,L,298.1,308.5,1498,49.4,5,0,0.02,0
3,L,298.2,308.6,1433,39.5,7,0,0.00,0
4,L,298.2,308.7,1408,40.0,9,0,0.01,0


In [6]:
X_all = df[
    [
        "Type",
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ]
]

threshold_tests = X_all.copy()

threshold_tests["Actual Failure"] = df["Machine failure"]

threshold_tests["Failure Probability"] = (
    model.predict_proba(X_all)[:, 1]
)

threshold_tests["Distance From Threshold"] = (
    threshold_tests["Failure Probability"] - FAILURE_THRESHOLD
).abs()

threshold_tests = threshold_tests.sort_values(
    "Distance From Threshold"
)

threshold_tests.head(10)

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Actual Failure,Failure Probability,Distance From Threshold
5391,L,302.7,312.2,1572,34.4,228,0,0.40,0.00
5399,H,302.8,312.4,1411,53.8,246,1,0.40,0.00
7847,L,300.2,311.6,1322,50.9,215,0,0.40,0.00
4118,L,302.0,310.6,1366,47.7,206,0,0.40,0.00
4851,L,303.7,312.1,1363,51.8,90,1,0.40,0.00
4114,L,302.0,310.6,1358,53.1,194,0,0.40,0.00
3568,L,301.8,310.6,1372,54.7,95,0,0.40,0.00
5590,M,302.5,311.9,1340,59.4,60,0,0.40,0.00
3906,L,302.3,311.2,1297,62.5,109,0,0.40,0.00
5593,L,302.6,311.9,1354,48.0,67,0,0.41,0.01


In [7]:
below_threshold = threshold_tests[
    threshold_tests["Failure Probability"] < FAILURE_THRESHOLD
].sort_values(
    "Failure Probability",
    ascending=False
)

below_threshold.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Actual Failure,Failure Probability,Distance From Threshold
5656,M,302.3,311.5,1316,47.2,4,0,0.39,0.01
7081,L,300.7,310.4,1366,52.6,205,0,0.38,0.02
5553,L,302.5,311.9,1306,59.7,172,0,0.38,0.02
4380,L,301.9,309.6,1410,49.0,223,0,0.38,0.02
7423,L,300.0,311.3,1407,47.3,208,0,0.38,0.02


## Application Testing Results

The completed Streamlit application was manually tested using observations
from the AI4I dataset. Known failure and non-failure observations were entered
through the application's user interface and the resulting probabilities and
classifications were compared with predictions generated directly from the
saved model pipeline.

Known failure observations produced high failure probabilities and were
correctly classified as potential machine failures, while known non-failure
observations produced low probabilities and were correctly classified as
non-failures.

Additional testing was performed around the application's 40% decision
threshold. Observations producing probabilities of 39%, 40%, and 41% were
tested. The 39% observation was classified as a non-failure, while the 40%
and 41% observations were classified as potential failures. This confirms
that the application correctly implements the intended `>= 0.40` decision
rule.

These tests also verified that the saved preprocessing and Random Forest
pipeline can successfully accept data from the Streamlit interface and
produce the same predictions observed during notebook testing.

In [1]:
# Get failure probabilities from the final model
y_prob = model.predict_proba(X_test)[:, 1]

# Apply the same threshold used by the Streamlit app
y_pred_40 = (y_prob >= 0.40).astype(int)

precision = precision_score(y_test, y_pred_40)
recall = recall_score(y_test, y_pred_40)
f1 = f1_score(y_test, y_pred_40)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Precision: {precision:.1%}")
print(f"Recall:    {recall:.1%}")
print(f"F1 Score:  {f1:.1%}")
print(f"ROC-AUC:   {roc_auc:.3f}")

NameError: name 'model' is not defined